# Construction du dataset Européennes 2024

En bonus des trois présidentielles, on ajoute les **européennes du 9 juin 2024**, qui sont
le scrutin national le plus récent. C'est utile pour le contexte 2027 : on y voit la montée
du RN, la percée de Glucksmann (PS) et le recul de LFI par rapport à la présidentielle.

⚠️ Attention : une européenne se compare mal à une présidentielle (abstention bien plus forte,
vote plus protestataire). À utiliser comme **contexte récent**, pas comme une 4e présidentielle.

Source : Ministère de l'Intérieur via data.gouv.fr (résultats par région).
Le fichier brut a un format "large" : 16 colonnes communes puis 38 listes, chacune sur 8 colonnes.

In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data_clean', exist_ok=True)

# lecture sans en-tête (on traite par position de colonne)
raw = pd.read_excel('data_brut/europeennes_2024_regions_raw.xlsx', header=None)
data = raw.iloc[1:].copy()   # on enlève la ligne d'en-tête
print('Brut :', raw.shape, '— 18 régions, 38 listes')

## Colonnes communes et listes principales

On garde les 8 plus grosses listes, repérées par leur numéro de panneau.

In [ ]:
# colonnes communes (par position)
df = pd.DataFrame()
df['Code_region'] = data[0].values
df['Region'] = data[1].values
df['Inscrits'] = pd.to_numeric(data[2], errors='coerce').values
df['Votants'] = pd.to_numeric(data[3], errors='coerce').values
df['Abstentions'] = pd.to_numeric(data[5], errors='coerce').values
df['Exprimes'] = pd.to_numeric(data[7], errors='coerce').values

# liste principale -> numéro de panneau (chaque liste = bloc de 8 colonnes à partir de la 16)
listes = {
    'RN': 5,            # Bardella - La France revient
    'Renaissance': 11,  # Hayer - Besoin d'Europe
    'PS_Glucksmann': 27,# Glucksmann - Réveiller l'Europe
    'LFI': 4,           # Aubry - LFI
    'LR': 18,           # Bellamy
    'EELV': 6,          # Toussaint - Europe Écologie
    'Reconquete': 3,    # Maréchal
    'PCF': 33,          # Léon - Gauche unie
}
for nom, panneau in listes.items():
    base = 16 + (panneau - 1) * 8
    df[nom] = pd.to_numeric(data[base + 4], errors='coerce').values   # +4 = colonne Voix

# pourcentages
for nom in listes:
    df[f'% {nom}'] = (df[nom] / df['Exprimes'] * 100).round(2)
df['% Participation'] = (df['Votants'] / df['Inscrits'] * 100).round(2)

print(df[['Region', '% RN', '% LFI', '% PS_Glucksmann', '% Renaissance']].to_string(index=False))

## Vérification et export

In [ ]:
# totaux nationaux (vérification vs résultats officiels)
exp = df['Exprimes'].sum()
print(f"LFI national  : {df['LFI'].sum()/exp*100:.2f} %  (officiel 9,89 %)")
print(f"RN national   : {df['RN'].sum()/exp*100:.2f} %  (officiel 31,37 %)")
print(f"PS national   : {df['PS_Glucksmann'].sum()/exp*100:.2f} %  (officiel 13,83 %)")

df.to_csv('data_clean/europeennes_2024_clean.csv', sep=';', index=False, encoding='utf-8-sig')
print('\n✅ Exporté : data_clean/europeennes_2024_clean.csv', df.shape)